In [ ]:
# HW07: Attention, BERTopic and DeepLatent

Remember that these homework work as a completion grade. **You can skip one section of this homework.**

## Attention

In [3]:
!pip install torch --upgrade
!pip install tiktoken

import math
import torch
import tiktoken
import numpy as np
import pandas as pd

text = "Your journey starts with the start"

#TODO Tokenize the sentence with tiktoken (GPT-2)
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(text)
tokens = tokenizer.decode(token_ids)

print("Token IDs:", token_ids)
print("Tokens:", tokens)

Token IDs: [7120, 7002, 4940, 351, 262, 923]
Tokens: Your journey starts with the start


In [4]:
!gdown 1avykMlpaYg_x6tzwF4hHoOkRxNGLkDrL
# https://drive.google.com/file/d/1avykMlpaYg_x6tzwF4hHoOkRxNGLkDrL/view?usp=share_link
glove = {}

glove = {}
with open (r'glove.6B.50d.txt', 'rb') as f:
  for line in f:
    parts = line.split()
    word = parts[0].decode('utf-8')
    vector = np.array(parts[1:], dtype=np.float32)
    glove[word] = vector

#TODO Convert each token into a GloVe vector. If a token is not found in GloVe, assign a zero vector
embedding_dim = 50
token_vectors = []

for tid, tok in zip(token_ids, tokens):
    # Strip leading/trailing whitespace and lowercase for GloVe lookup
    word = tok.strip().lower()
    if word in glove:
        vec = glove[word]
    else:
        # Zero vector for unknown tokens (OOV)
        vec = np.zeros(embedding_dim, dtype=np.float32)
        print(f"  Token '{tok}' (cleaned: '{word}') not found in GloVe → zero vector")
    token_vectors.append(vec)

#TODO Stack into the input embedding matrix X
X = np.stack(token_vectors)


Downloading...
From (original): https://drive.google.com/uc?id=1avykMlpaYg_x6tzwF4hHoOkRxNGLkDrL
From (redirected): https://drive.google.com/uc?id=1avykMlpaYg_x6tzwF4hHoOkRxNGLkDrL&confirm=t&uuid=71b06823-b376-40bd-9261-6d720c565e06
To: /content/glove.6B.50d.txt
100% 171M/171M [00:04<00:00, 35.9MB/s]
  Token ' ' (cleaned: '') not found in GloVe → zero vector


In [5]:
X

array([[ 0.11723  ,  1.0841   , -0.053105 ,  1.5335   , -0.14481  ,
        -0.57759  ,  0.34483  , -0.53862  , -1.0883   ,  0.85777  ,
         0.54502  ,  0.41332  ,  0.2146   , -0.62796  , -0.43957  ,
        -1.3131   , -0.33797  , -0.57519  ,  0.37648  ,  0.077038 ,
        -1.6088   , -0.71571  ,  0.76534  ,  0.37327  , -0.7326   ,
         0.32618  , -1.0791   , -0.10636  ,  0.73251  , -0.44364  ,
         1.6775   , -1.0514   , -0.96488  ,  0.070034 , -0.032693 ,
        -1.7435   , -0.078649 , -0.48295  ,  0.69566  ,  0.89638  ,
         0.59617  , -0.15894  ,  0.63727  , -1.0303   , -1.1533   ,
        -1.1079   , -0.13872  , -1.2205   ,  0.74533  ,  2.0112   ],
       [-0.043861 ,  1.3183   , -0.03715  ,  0.85478  ,  0.12212  ,
        -0.77384  ,  0.259    , -0.47676  , -0.33695  ,  0.48114  ,
         0.33945  ,  1.33     ,  0.2363   , -0.40044  , -0.060132 ,
        -0.090635 , -0.13512  , -0.24401  , -0.29643  ,  0.24748  ,
        -0.18947  , -0.31269  ,  0.81937  ,  0.

In [6]:
#TODO use the CausalAttention class to generate the context vectors

import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


context_length = X.shape[0]
d_in = X.shape[1]
d_out = X.shape[1]
dropout = 0.2
ca = CausalAttention(d_in, d_out, context_length, dropout)
X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(0)
context_vectors = ca(X_tensor)
print("Context Vectors Shape:", context_vectors.shape)

Context Vectors Shape: torch.Size([1, 6, 50])


In [7]:
context_vectors

tensor([[[-1.2263, -0.1694, -0.2412,  0.9008,  0.5996, -0.1979, -0.5777,
           0.1193, -0.2732,  0.9108,  1.3169, -0.7651, -0.7193, -0.3468,
          -0.0723, -0.4192, -0.8707,  0.2224,  0.1112, -1.0981, -0.3700,
           0.6111, -0.8739, -1.0398,  0.0669,  0.1129,  0.0776,  0.5926,
           0.0526, -0.9570, -0.2504,  1.2307,  0.6259,  0.7873,  1.0947,
          -0.4736, -0.2071,  0.3686,  0.0111, -0.1887,  0.0403,  1.4659,
          -0.8330,  1.0068, -0.1095,  0.2623,  0.2999,  0.2951,  0.1297,
           0.2182],
         [-0.9595, -0.1617,  0.1211,  0.7739,  0.2180, -0.4673, -0.2706,
           0.0742, -0.0913,  0.7562,  0.9338, -0.6943, -0.6991, -0.2690,
          -0.1527, -0.3504, -0.4877,  0.1369,  0.0120, -0.8170, -0.2991,
           0.7418, -0.8251, -0.7329,  0.1943,  0.1573, -0.0103,  0.7277,
           0.0287, -0.7466,  0.0681,  0.7784,  0.4497,  0.2705,  0.9943,
          -0.2315, -0.3243,  0.3210, -0.2165, -0.3641,  0.0034,  1.1336,
          -0.4837,  0.4808, -0.

## BERTopic on Congressional speeches

In [ ]:
# Load a random sample of speeches pronounced in the floor of the US Congress between 1994 and 2024
import pandas as pd
df = pd.read_csv('us_congress_speeches_sample.csv') # Found on the same Github
df['year'] = pd.to_datetime(df['date']).dt.year

print("Number of speeches: {}".format(len(df)))
df.head()

Number of speeches: 28731


,date,text,speaker_bioguide,chamber,party,doc_clean,year
0,2003-06-10,"Mr. ISRAEL. Mr. Speaker, I rise today to com...",I000057,House,Democrat,commend induction successful wrestling coach h...,2003
1,2012-05-08,"Mr. LANGEVIN. Mr. Speaker, Democrats are com...",L000559,House,Democrat,commit reduce deficit balanced way contrast br...,2012
2,2010-03-25,"Mr. BISHOP of Utah. Mr. Speaker, Lori Garver...",B001250,House,Republican,number administrator political give credit pri...,2010
3,1995-01-25,"Mr. KIM. Mr. Chairman, I rise today in suppo...",K000181,House,Republican,balanced budget small business tough run small...,1995
4,2003-05-22,"Mr. BAUCUS. Mr. President, sometime in the n...",B000243,Senate,Democrat,near future eye beholder go conference tax cut...,2003


In [ ]:
#!pip install bertopic[visualization]
from bertopic import BERTopic

#TODO fit a topic model (BERTopic) restricting to 10 topics

docs = df['doc_clean'].tolist()

topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True, nr_topics=10)
topics, probs = topic_model.fit_transform(docs)

In [ ]:
#TODO print out and visualize the top 5 words for each topic.

all_topics = topic_model.get_topics()
print("Top 5 words for each topic:")
for topic_id, words_with_scores in all_topics.items():
    # Excluding outlier topic
    if topic_id == -1:
        continue

    top_5_words = [word for word, score in words_with_scores[:5]]

    print(f"Topic {topic_id}: {top_5_words}")

fig = topic_model.visualize_barchart(top_n_topics=10)
fig.show()

# DeepLatent

Now we will estimate a topic model but accounting for covariates (metadata)

In [ ]:
#!pip install deeplatent
docs = df['doc_clean'].to_list()

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from deeplatent.corpus import Corpus
from deeplatent.models import GTM

vectorizer = CountVectorizer(
    ngram_range=(1, 1),
    max_df=0.95,
    min_df=0.001,
    stop_words="english",
    token_pattern=r"(?u)\b\w{3,}\b"  # only words with 3 or more letters
)

vectorizer.fit(docs)

print("Number of words in the vocabulary: {}".format(len(vectorizer.get_feature_names_out())))

# 2. Define modalities using the external vectorizer
modalities = {
    "text": {
        "column": "snippet",
        "views": {
            "bow": {
                "type": "bow",
                "vectorizer": vectorizer
            }
        }
    }
}

#TODO build the train dataset to estimate a GTM using as covariates the party, year and chamber for both prevalence and content.

train_dataset = Corpus(
    df=df,
    modalities=modalities,
    prevalence="~ C(party) + C(year)", #TODO add prevalence covs
    content="~ C(party) + C(year) + C(chamber)" #TODO add content covs
)

#TODO provide a brief intuition on why prevalence might matter by year and content might matter by party


In [ ]:
#TODO estimate a GTM (same specs as in the lab). Retrain to the amount of steps that you think necessary for the topics to make sense. Use 10 topics.
encoder_args = {
    "text_bow": {
        "hidden_dims": [256,128],
        "activation": "relu",
        "bias": True,
        "dropout": 0.1
    }
}

decoder_args = {
    "text_bow": {
        "hidden_dims": [128,256],
        "activation": "relu",
        "bias": True,
        "dropout": 0.1
    }
}

tm = GTM(
    train_dataset,
    n_topics=10,
    ae_type="wae", # WAE has very stable training and can use a Dirichlet prior, VAE has stronger theoretical guarantees but is prone to posterior collapse
    doc_topic_prior="logistic_normal",
    update_prior=True,
    encoder_args=encoder_args,
    decoder_args=decoder_args,
    batch_size=64,
    w_prior=1,
    return_best_model=True,
    num_steps=2000,
    print_every_n_steps=1000,
    print_topics=True,
    print_covariates=True
)

# Resume training if you think the topics need more refining
# tm.num_steps = tm.steps + 5000  # Train 5000 more steps
# tm.train(train_data)

print(
    "\n".join(
        [
            "{}: {}".format(str(k), str(v))
            for k, v in tm.get_topic_words(topK=5).items()
        ]
    )
)

#TODO Did the interpretability increased with respect to BERTopic?

In [ ]:
#TODO feed the topics (first 10 words) into the LLM of your choice (either using and API or manually) and ask it to interpret them

In [ ]:
#TODO Finally, do Republicans use different words by topic? Show the 10 most used words by topic for Republicans

#TODO Finally, do Republicans use different words by topic? Show the 10 most used words by topic for Republicans

print("Top 10 words per topic for Republicans: ")

print(
    pd.DataFrame(
        tm.get_topic_words(
            l_content_covariates=["Intercept", "C(party)[T.Republican]"]
        )
    )
)